# Module 20 — Processes that run and wait

A short module, and the only one in Part 5 with no framework in it. Everything from
here on is a program that **does not finish**: you start it, it waits, and your
terminal is occupied until you stop it.

Three things you have not needed until now: what a port is, what a blocking process
does to your shell, and how to get out of one. Plus the piece of the course's own
design that Part 5 depends on.

## 1. A port is a number a program asks for

A machine has one address and sixty-five thousand five hundred and thirty-five ports.
A server picks one and says "requests arriving here are for me"; a client says "connect
me to this address, at this number". `localhost:5000` is exactly that pair: the address
`127.0.0.1`, and port 5000 on it.

`socket` is the standard library, and the whole mechanism fits in three lines.

In [ ]:
import socket

listener = socket.socket()
listener.bind(("127.0.0.1", 0))  # port 0 means "any free one -- you choose"
chosen = listener.getsockname()[1]

print("the operating system gave me port", chosen)
listener.close()

`0` is the trick that module 16's and 17's servers both use: asking for any free port
means the tests cannot clash with each other or with whatever else you have running.
A real server asks for a fixed number, because clients have to know where to look.

The conventions you will meet in the next five modules:

| | |
| --- | --- |
| 5000 | Flask's default |
| 8000 | FastAPI/uvicorn's default |
| 8501 | Streamlit's default |
| 8888 | Jupyter's default — you have been using this since module 00 |
| below 1024 | reserved; binding one needs administrator rights |

None of them is special. They are agreements, and any free number above 1024 works.

## 2. Two programs, one port

A port can be held by one listener at a time. Ask for one that is taken and the bind
fails — which is the error you will actually meet in Part 5, when you start a second
Flask without stopping the first.

In [ ]:
import socket

first = socket.socket()
first.bind(("127.0.0.1", 0))
port = first.getsockname()[1]

second = socket.socket()
try:
    second.bind(("127.0.0.1", port))
    outcome = "worked"
except OSError as err:
    outcome = type(err).__name__

first.close()
second.close()

# What happens when the second program asks for a port the first one holds?
assert outcome == ...

In [ ]:
import socket

first = socket.socket()
first.bind(("127.0.0.1", 0))
port = first.getsockname()[1]

second = socket.socket()
try:
    second.bind(("127.0.0.1", port))
except OSError as err:
    print(type(err).__name__, "-", err.strerror)
    print("errno:", err.errno, "-- 48 on macOS, 98 on Linux, 10048 on Windows")

first.close()
second.close()

The message is `Address already in use`, and the number differs per operating system —
which is why code that has to react to it checks `errno.EADDRINUSE` rather than a
literal.

What it looks like when Flask says it:

```
OSError: [Errno 48] Address already in use
```

and the fix is almost always **the previous one is still running**. Find it and stop
it, or start this one on a different port:

```console
$ uv run flask --app app run --port 5001
```

## 3. A server does not return

Every program you have written so far ran, did something, and ended. A server calls
something like `app.run()` and that call **does not come back** until the server stops.

```python
app.run(port=5000)     # this line does not finish
print("never printed") # until you press Ctrl+C
```

Two consequences that surprise people the first time:

- **Your terminal is occupied.** The prompt does not come back, and typing does
  nothing useful. You need a second terminal to use `curl`, or a browser.
- **Nothing after that line runs.** Code you put below `app.run()` is unreachable
  until shutdown, which is not what "at the end of the program" usually means.

What is actually happening: the process is asleep in a system call, waiting for a
connection. It uses no processor time while it waits — a blocking process is not a
busy one.

In [ ]:
import subprocess
import sys
import time

# A stand-in for a server: a program that starts and then waits.
BLOCKING = """
import time
print("listening", flush=True)
time.sleep(30)
print("this line is not reached")
"""

process = subprocess.Popen([sys.executable, "-c", BLOCKING], stdout=subprocess.PIPE, text=True)
time.sleep(0.5)

print("the process is", "still running" if process.poll() is None else "finished")
print("and it has printed:", process.stdout.readline().strip())

process.terminate()
process.wait(timeout=5)

`process.poll()` returns `None` while a process is running and its exit code when it
has finished. That is how the cell above can tell.

## 4. Ctrl+C

Ctrl+C sends the process the signal **SIGINT**, and Python turns SIGINT into an
exception: `KeyboardInterrupt`. Which is why module 09 said that
`KeyboardInterrupt` comes off `BaseException` and not `Exception` — this is the moment
that mattered.

In [ ]:
import signal
import subprocess
import sys
import time

CATCHES = """
import time
print("listening", flush=True)
try:
    time.sleep(30)
except KeyboardInterrupt:
    print("shutting down cleanly", flush=True)
"""

process = subprocess.Popen([sys.executable, "-c", CATCHES], stdout=subprocess.PIPE, text=True)
time.sleep(0.5)

process.send_signal(signal.SIGINT)  # exactly what Ctrl+C sends
output, _ = process.communicate(timeout=5)

print(output.strip().replace("\n", " | "))
print("exit code:", process.returncode)

So `try: ... except KeyboardInterrupt:` around a server's run loop is how a program
shuts down tidily — closes its database connection, writes its last log line, and
exits 0.

Without it, the interpreter prints a traceback ending in `KeyboardInterrupt` and the
process ends with a **negative** exit code:

In [ ]:
import signal
import subprocess
import sys
import time

process = subprocess.Popen(
    [sys.executable, "-c", 'import time\nprint("listening", flush=True)\ntime.sleep(30)'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)
time.sleep(0.5)
process.send_signal(signal.SIGINT)
_, errors = process.communicate(timeout=5)

print("exit code:", process.returncode)
print("last line of stderr:", errors.strip().splitlines()[-1])

`-2` means "ended by signal 2", and signal 2 is SIGINT. A positive code is a value the
program chose; a negative one is a signal it did not survive. `128 + 2 = 130` is what
a shell reports for the same thing, which is why you will see both numbers.

**If Ctrl+C does not work**, there are three usual causes and one to be embarrassed
about:

- the process is stuck in a system call that does not return;
- something catches `KeyboardInterrupt` and ignores it — module 09's bare `except:`,
  which is exactly the bug it warned about;
- you are in a subshell or a debugger that swallows the key.

Then Ctrl+Z (suspend) and `kill %1`, or `kill <pid>` from another terminal. `kill -9`
sends SIGKILL, which cannot be caught and therefore cannot be tidy — a last resort,
not a habit.

## 5. Which address you bind to is a decision

`127.0.0.1` and `0.0.0.0` look like a detail and are the difference between a server
only you can reach and one the network can.

In [ ]:
import socket

print("127.0.0.1", "-- the loopback interface: this machine only, always available")
print("0.0.0.0  ", "-- every interface: reachable from anything that can route to you")
print()
print("this machine is called", socket.gethostname())
print("and resolves 'localhost' to", socket.gethostbyname("localhost"))

Flask, uvicorn and Streamlit all default to `127.0.0.1` for development, and that is
the right default: a half-finished application with `debug=True` on it should not be
answering the coffee shop's wifi. `debug=True` in particular gives anybody who can
reach the port an **interactive Python console** on a traceback page — which is
wonderful while you are writing and indefensible anywhere else.

The rule for Part 5: **`0.0.0.0` only when you mean it, and never together with
`debug=True`.**

In a container it is different — the container's `127.0.0.1` is not the host's, so a
service in Docker binds `0.0.0.0` and the container's network does the isolating.

## 6. What Part 5 is, and the package underneath it

Five modules, five frameworks, and **one task**: show these readings to a person.

| module | | what it makes |
| --- | --- | --- |
| 21 | Flask | a web page, with routes and a template |
| 22 | Streamlit | the same analysis, with no HTML and no routing |
| 23 | FastAPI | a JSON API, whose documentation is generated from your type hints |
| 24 | Tkinter | a desktop window, and an event loop |
| 25 | Textual | the same in a terminal |

Holding the task constant is the whole design. Five frameworks described one after
another would produce five sets of syntax; five frameworks solving the same problem
produce a comparison, and the answer to "when would I use which" falls out instead of
having to be asserted.

For that to be honest, the analysis underneath has to be **the same code**. It is:
`sensorreport` is an installed package in this repository, and every one of the five
modules imports it.

In [ ]:
from sensorreport import LIMIT, faults, load_readings, summarise

readings = load_readings()

print(len(readings), "readings,", sum(1 for r in readings if r.value is None), "unreadable")
print()

for summary in summarise(readings):
    print(
        f"{summary.location:<10}{summary.readings:>3}{summary.usable:>4}"
        f"{summary.mean:>8}{summary.highest:>7}{summary.faults:>3}"
    )

print()
print("above", LIMIT, ":", [(r.tag, r.value) for r in faults(readings)])

Those are the same numbers as module 18's pandas summary and module 19's SQL query,
computed a third way — with a `@dataclass` from module 12 and no dependency on
anything. Which is the point: the analysis is settled, so the five modules that follow
are about presentation and nothing else.

Note the import. No `sys.path` line, no relative path, from any directory: it works
because `pyproject.toml` lists `sensorreport` as a package of this project and
`uv sync` installed it. That is module 10's argument, made by the repository rather
than asserted in it.

In [ ]:
import sensorreport

print(sensorreport.__file__)
print([name for name in sensorreport.__all__])

---

`exercises/` is next: five files to fill in and two to think through. They are shorter
than usual — this module is a ramp, not a topic.

Module 21 is Flask: the same numbers, as a page in a browser.